# GGUF Colab Server (llama.cpp)

Google Colab GPU'sunda llama.cpp ile GGUF model serve et, Cloudflare Tunnel ile `api.ersamely.com` üzerinden eriş.

## Colab Secrets (gerekli)
- `HF_TOKEN` — HuggingFace erişimi
- `CF_TUNNEL_TOKEN` — Cloudflare named tunnel
- `VLLM_API_KEY` — API erişim anahtarı

## Kullanım
1. **A**: Kurulum (bir kere)
2. **B1**: Model ayarla
3. **B2**: Model indir + server başlat
4. **C1**: Tunnel bağla
5. **C2**: Keepalive (opsiyonel)

---
# A) İlk Kurulum (bir kere)

In [ ]:
import os
import subprocess

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    subprocess.run(
        ['huggingface-cli', 'login', '--token', hf_token, '--add-to-git-credential'],
        capture_output=True, text=True, check=True
    )
    print('\u2705 HuggingFace login')
except KeyError:
    print('\u26a0\ufe0f HF_TOKEN tanımlı değil')
except Exception as e:
    print(f'\u26a0\ufe0f HF login atlandı: {e}')

!CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python[server] --force-reinstall --no-cache-dir 2>&1 | tail -5
!pip -q install httpx huggingface_hub 2>&1 | tail -3

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

!nvidia-smi | head -12

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

MODEL_DIR = '/content/models'
LOG_DIR = '/content/logs'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

---
# B) Model Başlat

In [ ]:
from google.colab import userdata

HF_REPO      = 'bartowski/Qwen3-32B-GGUF'
GGUF_FILE    = 'Qwen3-32B-Q4_K_M.gguf'
GGUF_SPLIT   = False
N_GPU_LAYERS = -1
N_CTX        = 32768
PORT         = 8090

# API Key — kimliksiz erişimi engeller
API_KEY      = userdata.get('VLLM_API_KEY')

MODEL_PATH    = f'{MODEL_DIR}/{GGUF_FILE}'
VLLM_BASE_URL = f'http://localhost:{PORT}'
TUNNEL_URL    = 'https://api.ersamely.com'

print(f'Repo:         {HF_REPO}')
print(f'File:         {GGUF_FILE}')
print(f'n_gpu_layers: {N_GPU_LAYERS}')
print(f'n_ctx:        {N_CTX}')
print(f'API Key:      {API_KEY[:8]}...')
print(f'Tunnel:       {TUNNEL_URL}')

In [ ]:
import subprocess
import time

import requests

print('\U0001f6d1 Önceki server durduruluyor...')
subprocess.run(['pkill', '-f', 'llama_cpp.server'], capture_output=True)
time.sleep(2)

# Model indir
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download, list_repo_files
    if GGUF_SPLIT:
        base = GGUF_FILE.rsplit('-00001-of-', 1)[0] if '-00001-of-' in GGUF_FILE else GGUF_FILE.rsplit('.', 1)[0]
        all_files = list_repo_files(HF_REPO)
        parts = sorted(f for f in all_files if f.startswith(base) and f.endswith('.gguf'))
        print(f'\U0001f4e5 Split: {len(parts)} parça')
        for p in parts:
            if not os.path.exists(f'{MODEL_DIR}/{p}'):
                print(f'   \u2193 {p}')
                hf_hub_download(repo_id=HF_REPO, filename=p, local_dir=MODEL_DIR)
    else:
        print(f'\U0001f4e5 İndiriliyor: {GGUF_FILE}')
        hf_hub_download(repo_id=HF_REPO, filename=GGUF_FILE, local_dir=MODEL_DIR)
    print('\u2705 İndirildi')
else:
    print(f'\u2705 Mevcut: {MODEL_PATH}')

# Server başlat (llama.cpp native API key desteği yok, vLLM gibi --api-key parametresi yok)
LOG_PATH = f'{LOG_DIR}/llama.log'
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', MODEL_PATH,
    '--n_gpu_layers', str(N_GPU_LAYERS),
    '--n_ctx', str(N_CTX),
    '--host', '0.0.0.0',
    '--port', str(PORT),
    '--api_key', API_KEY,
]

with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL
    )

print(f'\U0001f680 Server başlatıldı (PID: {proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 600:
    try:
        if requests.get(f'{VLLM_BASE_URL}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if proc.poll() is not None:
        print(f'\u274c Çöktü! (exit: {proc.returncode})')
        try:
            with open(LOG_PATH) as f: print(f.read()[-1000:])
        except OSError: pass
        break
    print(f'[{int(time.time()-t0):3d}s] \u23f3 Başlatılıyor')
    time.sleep(5)

if ready:
    print(f'\n\u2705 Server hazır! ({int(time.time()-t0)}s)')
    try:
        r = requests.post(
            f'{VLLM_BASE_URL}/v1/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}'},
            json={'model': MODEL_PATH, 'messages': [{'role': 'user', 'content': 'Say hello'}], 'max_tokens': 32},
            timeout=120,
        )
        if r.status_code == 200:
            txt = r.json()['choices'][0]['message'].get('content', 'No content')
            print(f'   \U0001f4ac {txt[:150]}')
        else:
            print(f'   \u26a0\ufe0f HTTP {r.status_code}')
    except requests.RequestException as e:
        print(f'   \u26a0\ufe0f {e}')
else:
    print('\n\u274c Timeout!')

---
# C) Cloudflare Tunnel

Named tunnel: `api.ersamely.com` \u2192 `localhost:8090`

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

token = userdata.get('CF_TUNNEL_TOKEN')

CF_LOG = f'{LOG_DIR}/cloudflared.log'
with open(CF_LOG, 'w') as f:
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
        stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    )

print(f'\U0001f310 Tunnel başlatıldı (PID: {cf_proc.pid})')
time.sleep(5)

try:
    r = requests.get(f'{TUNNEL_URL}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=10)
    if r.status_code == 200:
        print(f'\u2705 Tunnel aktif: {TUNNEL_URL}')
        print(f'\n   Open WebUI:')
        print(f'   URL: {TUNNEL_URL}/v1')
        print(f'   API Key: (Colab Secrets > VLLM_API_KEY)')
    else:
        print(f'\u26a0\ufe0f HTTP {r.status_code}')
except requests.RequestException:
    if cf_proc.poll() is None:
        print(f'\u2705 Tunnel çalışıyor: {TUNNEL_URL}')
        print('   (Server hazır olunca erişilebilir)')
    else:
        print('\u274c Tunnel başarısız!')
        try:
            with open(CF_LOG) as f: print(f.read()[-500:])
        except OSError: pass

In [ ]:
import time
from datetime import datetime, timezone

import requests

print(f'Canlı tutma: {TUNNEL_URL}')
print('Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'{VLLM_BASE_URL}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    try:
        tunnel_ok = requests.get(f'{TUNNEL_URL}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=10).status_code == 200
    except requests.RequestException:
        tunnel_ok = False

    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    l = '\u2705' if local_ok else '\u274c'
    t = '\u2705' if tunnel_ok else '\u274c'
    print(f'{now} | Server: {l} | Tunnel: {t}')
    time.sleep(30)